# inference

> Turn a trained model into predictions: probability maps (optionally test-time augmented), label decoding, and an operating-threshold search for binary tasks.

`predict` returns per-class probabilities for both binary (sigmoid) and multi-class (softmax). `decode` collapses them to a label mask. `find_operating_threshold` sweeps the binary decision threshold to hit a target recall, which matters when a miss is far costlier than a false alarm.

In [1]:
#| default_exp inference

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
import cv2

from typing import Dict, List, Optional, Sequence

import numpy as np
import torch

from segmentation_tools_pt.stitching import gaussian_weight_map


In [4]:
#| exporti
from segmentation_tools_pt.metrics import ObjectMetrics

## Predict probabilities

One forward pass returns `(C, H, W)` probabilities. With `tta=True` we average predictions over the four 90-degree rotations, rotating each prediction back before averaging. TTA trades compute for a small, usually-positive accuracy bump on thin structures.

`task` disambiguates `C > 1`: `multiclass` (default) uses softmax -- classes are mutually exclusive, so probabilities sum to 1 across channels. `multilabel` uses an independent sigmoid per channel instead, so overlapping classes (e.g. `void` inside `chip`) aren't forced to compete for probability mass. `binary` (`C == 1`) always uses sigmoid regardless of `task`.

### Shared activation rule

Both `predict` and `predict_tiled` must turn logits into probabilities the same way: **sigmoid** when the head is binary (`C == 1`) or `task='multilabel'` (channels independent, overlapping classes allowed), **softmax** across channels for `multiclass` (mutually exclusive classes), and a pure **passthrough** when `activate=False`. That rule used to live twice -- once inside `predict`'s `forward`, once inside `predict_tiled`'s `activate_probs` -- so the two copies could drift and the same model would get different probabilities depending on image size (issue #25).

Plan (experiment -> export -> test):

1. Experiment: run the rule inline on synthetic logits for the four pinned cases (binary sigmoid, multilabel sigmoid, multiclass softmax, logits passthrough).
2. Promote it to `_activate(logits, task, activate)`; both public functions delegate to it. The leading underscore keeps it private -- out of `__all__` -- so callers keep using `predict` / `predict_tiled`.
3. Test the four cases against the export, on the same synthetic logits.

In [ ]:
# experiment: the activation rule, inline, on synthetic logits — not exported yet
logits_1c = torch.tensor([[[-2.0, 0.0, 3.0]]])          # (1, 1, 3) binary head
logits_3c = torch.tensor([[[ 2.0, 0.0, -1.0],
                           [ 0.0, 1.0,  2.0],
                           [ 1.0, 1.0,  1.0]]])          # (1, 3, 3) 3-class head

def _activate_proto(logits, task, activate):
    if not activate: return logits
    return torch.sigmoid(logits) if (logits.shape[1] == 1 or task == 'multilabel') else torch.softmax(logits, dim=1)

cases = [
    ('binary',      _activate_proto(logits_1c, None, True)),
    ('multilabel',  _activate_proto(logits_3c, 'multilabel', True)),
    ('multiclass',  _activate_proto(logits_3c, None, True)),
    ('passthrough', _activate_proto(logits_3c, None, False)),
]
for name, a in cases:
    print(f'{name:12s} shape={tuple(a.shape)} range=[{a.min():+.3f}, {a.max():+.3f}]')

# verify each branch
assert torch.equal(_activate_proto(logits_1c, None, True), torch.sigmoid(logits_1c))
assert torch.equal(_activate_proto(logits_1c, 'multiclass', True), torch.sigmoid(logits_1c))  # C==1 wins over task
assert torch.equal(_activate_proto(logits_3c, 'multilabel', True), torch.sigmoid(logits_3c))
mc = _activate_proto(logits_3c, None, True)
assert torch.allclose(mc.sum(1), torch.ones(1, 3))  # softmax sums to 1 across channels
assert torch.equal(_activate_proto(logits_3c, None, False), logits_3c)
print('activation rule verified for all four cases')

In [ ]:
#| export
def _activate(
    logits: torch.Tensor,  # (B, C, H, W) raw model output
    task: str,             # 'binary' | 'multiclass' | 'multilabel'; None infers binary (C==1) or multiclass (C>1)
    activate: bool,        # if False, return logits untouched
) -> torch.Tensor:
    """Sigmoid if `C==1` or `task='multilabel'`, softmax for multiclass, logits passthrough if `activate=False`.

    The one activation rule shared by `predict` and `predict_tiled` (issue #25), kept
    private so the two public entry points can never drift apart.

    Example:
        >>> _activate(torch.tensor([[[-2.0, 3.0]]]), None, True).round(decimals=3).tolist()
        [[[0.119, 0.953]]]
    """
    if not activate: return logits
    return torch.sigmoid(logits) if (logits.shape[1] == 1 or task == 'multilabel') else torch.softmax(logits, dim=1)

In [ ]:
#| test
# test: the four pinned activation cases (issue #25) — one rule, two callers
assert torch.equal(_activate(logits_1c, None, True), torch.sigmoid(logits_1c))            # binary C==1 -> sigmoid
assert torch.equal(_activate(logits_1c, 'multiclass', True), torch.sigmoid(logits_1c))    # C==1 wins over task
assert torch.equal(_activate(logits_3c, 'multilabel', True), torch.sigmoid(logits_3c))    # multilabel -> per-channel sigmoid
mc = _activate(logits_3c, None, True)
assert torch.allclose(mc.sum(1), torch.ones(1, 3))                                        # multiclass -> softmax over channels
assert torch.equal(_activate(logits_3c, None, False), logits_3c)                          # activate=False -> logits untouched
print('_activate ✓  sigmoid / softmax / passthrough pinned')

In [5]:
#| export
@torch.no_grad()
def predict(
    model,                 # trained model returning (B, C, H, W) logits
    image: torch.Tensor,   # a single (C, H, W) input
    device: torch.device,  # cpu or cuda
    tta: bool = False,     # average over 90-degree rotations
    n_augs: int = 4,       # number of rotations when tta=True
    activate: bool = True, # apply sigmoid/softmax to logits
    task: str = None,      # 'binary' | 'multiclass' | 'multilabel'; None infers binary (C==1) or multiclass (C>1)
) -> torch.Tensor:
    """Return `(C, H, W)` probabilities: sigmoid if `C==1` or `task='multilabel'`, else softmax."""
    model.eval()
    x = image.unsqueeze(0).to(device)

    def forward(t: torch.Tensor) -> torch.Tensor:
        return _activate(model(t), task, activate)

    if not tta:
        return forward(x).squeeze(0)
    preds = []
    for k in range(n_augs):
        pr = forward(torch.rot90(x, k, dims=[2, 3]))
        preds.append(torch.rot90(pr, -k, dims=[2, 3]))
    return torch.stack(preds).mean(0).squeeze(0)

In [6]:
#| export
def decode(
    probs: torch.Tensor,     # (C, H, W) probabilities
    threshold=0.5,            # decision threshold: float (binary/multilabel-shared), or per-channel Sequence[float] for multilabel
    task: str = None,         # 'binary' | 'multiclass' | 'multilabel'; None infers binary (C==1) or multiclass (C>1)
) -> torch.Tensor:
    """Collapse probabilities to a label mask.

    `binary`/`multiclass` (default behavior, unchanged) return `(H, W)`: a
    thresholded `{0,1}` mask for `C==1`, or an argmax label map for `C>1`.
    `multilabel` returns `(C, H, W)` -- classes can overlap, so there is no
    single `(H,W)` label map to collapse to; each channel is thresholded
    independently instead.

    Example:
        >>> decode(torch.tensor([[[0.2, 0.9]]])).tolist()
        [[0, 1]]
    """
    if task == 'multilabel':
        c = probs.shape[0]
        # duck-type "sequence of per-channel thresholds" rather than isinstance(list, tuple):
        # Hydra/OmegaConf configs hand back ListConfig, not a plain list, for a YAML list.
        is_seq = hasattr(threshold, '__len__') and not isinstance(threshold, (str, bytes))
        t = list(threshold) if is_seq else [threshold] * c
        return torch.stack([probs[k] >= t[k] for k in range(c)]).long()
    if probs.shape[0] == 1:
        return (probs[0] >= threshold).long()
    return probs.argmax(0)

In [7]:
#| test
# test: shapes for binary and multi-class, with and without TTA
import torch.nn as nn
dev = torch.device('cpu')
mb = nn.Conv2d(3, 1, 3, padding=1)
p = predict(mb, torch.rand(3, 16, 16), dev)
assert p.shape == (1, 16, 16)
assert predict(mb, torch.rand(3, 16, 16), dev, tta=True).shape == (1, 16, 16)
assert decode(p).shape == (16, 16)
mc = nn.Conv2d(3, 3, 3, padding=1)
pc = predict(mc, torch.rand(3, 16, 16), dev)
assert pc.shape == (3, 16, 16) and decode(pc).shape == (16, 16)

In [8]:
#| test
# test: multilabel predict/decode -- independent per-channel sigmoid, no softmax, overlap preserved
m3 = nn.Conv2d(3, 3, 3, padding=1)
p3 = predict(m3, torch.rand(3, 16, 16), dev, task='multilabel')
assert p3.shape == (3, 16, 16)
assert not torch.allclose(p3.sum(0), torch.ones(16, 16))  # not softmax-normalised

d3 = decode(p3, task='multilabel')
assert d3.shape == (3, 16, 16)  # unlike multiclass decode, NOT collapsed to (H,W)

# per-channel thresholds actually take effect
probs_fixed = torch.full((3, 4, 4), 0.6)
d_low = decode(probs_fixed, threshold=0.5, task='multilabel')
d_high = decode(probs_fixed, threshold=[0.5, 0.7, 0.5], task='multilabel')
assert bool(d_low[1].all()) and not bool(d_high[1].any())  # channel 1: 0.6>=0.5 fires, 0.6>=0.7 doesn't
print('multilabel predict/decode ✓  independent sigmoid, (C,H,W) decode, per-channel thresholds')

multilabel predict/decode ✓  independent sigmoid, (C,H,W) decode, per-channel thresholds


## Hierarchy post-process (multilabel)

`decode(task='multilabel')` thresholds each channel independently, so a
trained model can still predict a `void`/`lifted` pixel outside its `chip`
footprint even with `losses.hierarchy_regularizer_loss` nudging it during
training (that's a soft penalty, not a guarantee). `apply_hierarchy` hard-
enforces the containment relationship at inference time: any sub-channel
pixel not also covered by its super-channel is zeroed out.

In [9]:
#| export
def apply_hierarchy(
    masks: torch.Tensor,  # (C, H, W) or (B, C, H, W) decoded/binary multilabel masks
    pairs,                  # [(sub_idx, super_idx), ...], e.g. [(1, 0), (2, 0)] for (void,chip),(lifted,chip)
) -> torch.Tensor:
    """Zero out sub-channel pixels not covered by their super-channel. Returns a new tensor."""
    if not pairs:
        return masks
    masks = masks.clone()
    c_dim = -3  # channel is always third-from-last: (...,C,H,W)
    for s, p in pairs:
        sub = masks.select(c_dim, s)
        sup = masks.select(c_dim, p)
        if masks.dtype == torch.bool:
            sub &= sup
        else:
            sub.mul_(sup)
    return masks

In [10]:
#| test
# test: apply_hierarchy zeroes sub-channel pixels outside their super-channel, leaves the rest alone
chip = torch.zeros(4, 4); chip[1:3, 1:3] = 1
void = torch.zeros(4, 4); void[0:2, 0:2] = 1  # only (1,1) is inside chip; (0,0),(0,1),(1,0) leak outside
masks = torch.stack([chip, void]).long()  # (C=2, H, W): 0=chip, 1=void

fixed = apply_hierarchy(masks, pairs=[(1, 0)])
assert fixed[1, 1, 1] == 1        # inside chip -> kept
assert fixed[1, 0, 0] == 0        # outside chip -> zeroed
assert torch.equal(fixed[0], masks[0])  # super-channel (chip) itself is untouched
assert masks[1, 0, 0] == 1        # original input not mutated (clone, not in-place on the caller's tensor)

# batched (B, C, H, W) works the same way
batched = masks.unsqueeze(0).repeat(2, 1, 1, 1)
fixed_b = apply_hierarchy(batched, pairs=[(1, 0)])
assert fixed_b.shape == (2, 2, 4, 4)
assert bool((fixed_b[:, 1, 0, 0] == 0).all())

# empty pairs -> no-op, same tensor object back (cheap default)
assert apply_hierarchy(masks, []) is masks

# also works on a bool mask (as produced by decode(..., task='multilabel'))
bool_masks = masks.bool()
fixed_bool = apply_hierarchy(bool_masks, [(1, 0)])
assert fixed_bool.dtype == torch.bool and bool(fixed_bool[1, 0, 0]) is False
print('apply_hierarchy ✓  hard-enforces containment, (C,H,W) + (B,C,H,W) + bool, no input mutation')

apply_hierarchy ✓  hard-enforces containment, (C,H,W) + (B,C,H,W) + bool, no input mutation


## Class-targeted morph post-process (multilabel)

A per-channel sigmoid decoder can leave tiny speckle false alarms in one class
channel (e.g. `void`) that nothing during training punishes — deliberately so:
the loss and the raw train metrics must judge the model as-is. The cleanup
therefore belongs strictly to the inference/production decode step, and it is
opt-in config (`postprocess.class_morph.*`, master switch default **false**,
resolved by `hydra_pipeline.resolve_class_morph_cfg`).

`apply_class_morph` cleans **one named class channel** with classical
morphology: an opening (erode -> dilate, kills thin speckles) followed by a
connected-component min-area filter (kills small blobs that survived the
opening). All other channels pass through untouched. Fixed decode order:

> threshold decode (`decode`) -> class morph (`apply_class_morph`) -> hierarchy (`apply_hierarchy`)

Plan:
1. Experiment: a decoded 3-channel mask with 1-px speckles in one channel — verify opening + min-area clean only that channel.
2. Export `apply_class_morph`: shape-generic `(C,H,W)`/`(B,C,H,W)`, no input mutation, clear error on bad `class_idx`.
3. Test the acceptance criteria, including the fixed morph-then-hierarchy order.

In [11]:
# experiment: a decoded (C,H,W) mask with 1-px speckles in the void channel only
import cv2
import numpy as np

chip = torch.ones(12, 12).long()
void = torch.zeros(12, 12).long()
void[4:8, 4:8] = 1      # real 16-px blob
void[0, 0] = 1          # 1-px speckle
void[11, 11] = 1        # 1-px speckle
lifted = torch.zeros(12, 12).long(); lifted[4:8, 4:8] = 1

# inline logic first: opening + min-area on the void channel by hand
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
m = (void.numpy() > 0).astype(np.uint8)
m = cv2.erode(m, kernel, borderValue=0)   # outside-image = background
m = cv2.dilate(m, kernel, borderValue=0)
n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
for lbl in range(1, n):
    if stats[lbl, cv2.CC_STAT_AREA] < 8:
        m[labels == lbl] = 0
print(f'void fg: before={int(void.sum())} after={int(m.sum())}')  # expect 18 -> 16
assert int(m.sum()) == 16 and m[4:8, 4:8].all()  # speckles gone, blob intact

void fg: before=18 after=16


In [12]:
#| export
def apply_class_morph(
    masks: torch.Tensor,   # (C, H, W) or (B, C, H, W) decoded/binary multilabel masks
    class_idx: int,        # channel index of the class to clean (resolve class_name via class_names first)
    kernel_size: int = 3,  # opening structuring element side (odd); kills thin speckles
    min_area: int = 16,    # drop connected components smaller than this many pixels (1 = keep all)
) -> torch.Tensor:
    """Morph-clean ONE class channel (opening + min-area); other channels pass through untouched.

    Inference/production decode only — never applied to the training loss or to
    raw train metrics (those must judge the model as-is). Fixed decode order:
    `decode` -> `apply_class_morph` -> `apply_hierarchy`.

    Example:
        >>> cleaned = apply_class_morph(decoded, class_idx=1, kernel_size=3, min_area=16)
    """
    c_dim = -3  # channel is always third-from-last: (..., C, H, W)
    n_classes = masks.shape[c_dim]
    if not 0 <= class_idx < n_classes:
        raise ValueError(
            f'class_idx={class_idx} is out of range for a {n_classes}-channel mask. '
            'Resolve postprocess.class_morph.class_name against data.class_names first '
            '(see hydra_pipeline.resolve_class_morph_cfg).'
        )
    if kernel_size < 1:
        raise ValueError(f'kernel_size must be >= 1, got {kernel_size}')
    if min_area < 1:
        raise ValueError(f'min_area must be >= 1, got {min_area}')
    masks = masks.clone()
    ch = masks.select(c_dim, class_idx)  # view into the clone; writes below land in the clone
    items = ch if ch.dim() > 2 else ch.unsqueeze(0)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    for i, item in enumerate(items):
        m = (item.numpy() > 0).astype(np.uint8)
        # opening as explicit erode->dilate with outside-image = background:
        # border-touching speckles are removable, at the cost of real objects
        # touching the border losing up to kernel_size // 2 px at the boundary
        m = cv2.erode(m, kernel, borderValue=0)
        m = cv2.dilate(m, kernel, borderValue=0)
        if min_area > 1:
            n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
            for lbl in range(1, n):
                if stats[lbl, cv2.CC_STAT_AREA] < min_area:
                    m[labels == lbl] = 0
        items[i] = torch.from_numpy(m.astype(bool)).to(masks.dtype)
    return masks

In [13]:
#| test
# test: issue #19 acceptance — named channel cleaned, others untouched, no mutation, clear errors
chip = torch.ones(12, 12).long()
void = torch.zeros(12, 12).long(); void[4:8, 4:8] = 1; void[0, 0] = 1; void[11, 11] = 1
lifted = torch.zeros(12, 12).long(); lifted[4:8, 4:8] = 1
masks = torch.stack([chip, void, lifted])  # (C=3, H, W): 0=chip, 1=void, 2=lifted

cleaned = apply_class_morph(masks, class_idx=1, kernel_size=3, min_area=8)
assert int(cleaned[1].sum()) == 16 and bool(cleaned[1][4:8, 4:8].all())  # speckles gone, blob kept
assert torch.equal(cleaned[0], masks[0]) and torch.equal(cleaned[2], masks[2])  # others untouched
assert int(masks[1].sum()) == 18  # caller's tensor not mutated

# batched (B, C, H, W) behaves identically per batch item
batched = masks.unsqueeze(0).repeat(2, 1, 1, 1)
cleaned_b = apply_class_morph(batched, class_idx=1, kernel_size=3, min_area=8)
assert cleaned_b.shape == (2, 3, 12, 12)
assert int(cleaned_b[0, 1].sum()) == 16 and int(cleaned_b[1, 1].sum()) == 16

def _raises(fn, msg):
    try:
        fn()
        raise AssertionError(f'expected ValueError: {msg}')
    except ValueError as e:
        assert msg in str(e), str(e)

_raises(lambda: apply_class_morph(masks, class_idx=5), 'class_idx=5 is out of range')
_raises(lambda: apply_class_morph(masks, class_idx=1, kernel_size=0), 'kernel_size must be >= 1')
_raises(lambda: apply_class_morph(masks, class_idx=1, min_area=0), 'min_area must be >= 1')

# fixed order: threshold decode -> class morph (named class) -> hierarchy.
# Morph runs on the SUPER channel (chip): its small speckle dies, and hierarchy
# then drops the void pixels that lived under it. Reversed order would strand
# those void pixels inside the (no longer existing) chip speckle.
probs = torch.full((3, 12, 12), 0.1)
probs[0, 2:10, 2:10] = 0.9  # chip main blob
probs[0, 0:2, 0:2] = 0.9    # separate small chip speckle (decodes on, opening kills it)
probs[1, 0:2, 0:2] = 0.9    # void only inside that chip speckle (subset, no leak)
probs[2, 4:8, 4:8] = 0.9    # lifted inside the chip main blob
decoded = decode(probs, threshold=0.5, task='multilabel')
morphed = apply_class_morph(decoded, class_idx=0, kernel_size=3, min_area=8)
final = apply_hierarchy(morphed, pairs=[(1, 0), (2, 0)])
assert not bool(final[0, 0:2, 0:2].any())  # chip speckle removed by morph
assert bool(final[0, 2:10, 2:10].all())    # chip main blob survives opening
assert not bool(final[1].any())            # void died with its super: morph ran BEFORE hierarchy
assert bool(final[2, 4:8, 4:8].all())      # untouched channel survives the full chain
# master switch off = skip apply_class_morph entirely: nothing cleans the speckle
no_morph = apply_hierarchy(decoded, pairs=[(1, 0), (2, 0)])
assert bool(no_morph[1, 0:2, 0:2].all())  # void speckle survives under the chip speckle
print('apply_class_morph ✓  named channel cleaned, others untouched, batched, errors, morph→hierarchy order')

apply_class_morph ✓  named channel cleaned, others untouched, batched, errors, morph→hierarchy order


## Operating-threshold search (binary)

Sweep the sigmoid threshold over a validation loader and report pixel recall/precision/IoU plus object-level detection at each. We recommend the **highest** threshold that still meets `target_recall`: highest precision among the options that do not start missing targets.

In [14]:
#| export
@torch.no_grad()
def find_operating_threshold(
    model,                          # trained binary model
    val_loader,                     # validation DataLoader of {'image','mask'}
    device: torch.device,           # cpu or cuda
    target_recall: float = 0.99,    # minimum pixel recall to accept
    overlap_thresh: float = 0.5,    # object overlap fraction for detection
    thresholds: Optional[Sequence[float]] = None,  # thresholds to scan
) -> Dict:
    """Scan thresholds; return the recommended one plus the full sweep."""
    model.eval()
    thresholds = list(thresholds) if thresholds is not None else list(np.arange(0.05, 0.95, 0.05))
    prob_imgs, gt_imgs = [], []
    for batch in val_loader:
        images = batch['image'].to(device)
        p = torch.sigmoid(model(images))[:, 0].cpu().numpy()
        prob_imgs.append(p)
        gt_imgs.append((batch['mask'].numpy() > 0).astype(np.uint8))
    probs = np.concatenate(prob_imgs)        # (N, H, W)
    gts = np.concatenate(gt_imgs)            # (N, H, W)
    pf, gf = probs.reshape(-1), gts.reshape(-1).astype(bool)
    sweep = []
    for t in thresholds:
        pred = pf >= t
        tp = int((pred & gf).sum()); fp = int((pred & ~gf).sum()); fn = int((~pred & gf).sum())
        obj = ObjectMetrics(overlap_thresh)
        for pimg, gimg in zip(probs, gts):
            obj.update((pimg >= t).astype(np.uint8), gimg)
        od = obj.compute()
        sweep.append({'threshold': round(float(t), 2),
                      'recall': round(tp / (tp + fn + 1e-9), 4),
                      'precision': round(tp / (tp + fp + 1e-9), 4),
                      'iou': round(tp / (tp + fp + fn + 1e-9), 4),
                      'detection_rate': round(od['detection_rate'], 4),
                      'missed': od['missed'], 'false_alarms': od['false_alarms']})
    ok = [r for r in sweep if r['recall'] >= target_recall]
    best = max(ok, key=lambda r: r['threshold']) if ok else max(sweep, key=lambda r: r['recall'])
    return {'recommended_threshold': best['threshold'], 'achieved_recall': best['recall'],
            'achieved_precision': best['precision'], 'sweep': sweep}

In [15]:
#| test
# test: threshold search returns a valid recommendation on a tiny val loader
from torch.utils.data import Dataset, DataLoader

class _Val(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        m = torch.zeros(16, 16, dtype=torch.long); m[4:12, 4:12] = 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': str(i)}

res = find_operating_threshold(mb, DataLoader(_Val(), batch_size=2), dev, target_recall=0.0, thresholds=[0.3, 0.5, 0.7])
assert 0.0 < res['recommended_threshold'] < 1.0 and len(res['sweep']) == 3

## Batched tile inference (large images)

`predict` runs one view at a time — fine for crops, but a full inspection image
(thousands of pixels per side) needs sliding-window tiling, and running **all** tiles in
a single forward pass OOMs while running them one-by-one leaves the GPU idle between
tile loads. `predict_tiled` closes that gap: tiles are forwarded in chunks of
`inference_batch_size` and the chunk outputs are stitched back with Gaussian-weighted
blending, with the same activation semantics as `predict` (sigmoid for binary/multilabel,
softmax for multiclass).

Plan (experiment → verify → export):

1. Tile-start math: clamped sliding window (last start shifted back so every tile fits),
   proven to cover every pixel including images smaller than one tile (zero-padded up).
2. Batched forward + weighted accumulation: verify `inference_batch_size=1` gives
   **bit-identical** results to any larger value on a synthetic model — the knob trades
   memory against throughput only, never accuracy.
3. Promote to `predict_tiled`; edge cases: multilabel / multiclass heads, `activate=False`
   logits passthrough, `overlap >= tile_size` and `inference_batch_size < 1` fail clearly.

Production tie-in (issue #20): the same knob is the `inference.inference_batch_size` config
surface, and a curated-encoder ONNX smoke covering the tiled FP32 path lives in
`nbs/13_export.ipynb`.

In [16]:
# experiment: tile-start math (clamped sliding window) — not exported yet
def _tile_starts_proto(extent, tile_size, overlap):
    if not (0 <= overlap < tile_size):
        raise ValueError(f'overlap must satisfy 0 <= overlap < tile_size ({tile_size}), got {overlap}')
    stride = tile_size - overlap
    if extent <= tile_size: return [0]
    n = -(-(extent - tile_size) // stride) + 1  # ceil((extent - tile_size) / stride) + 1 windows
    return sorted({min(i * stride, extent - tile_size) for i in range(n)})

for extent, ts, ov in [(256, 64, 16), (70, 64, 16), (64, 64, 16), (10, 64, 16), (130, 64, 0)]:
    starts = _tile_starts_proto(extent, ts, ov)
    padded = max(extent, ts)
    covered = torch.zeros(padded, dtype=torch.bool)
    for s in starts: covered[s:s + ts] = True
    assert covered.all() and starts[0] == 0, (extent, ts, ov, starts)
    print(f'extent={extent:4d} tile={ts} overlap={ov:2d} -> {len(starts)} starts {starts}')
print('tile math covers every pixel')

extent= 256 tile=64 overlap=16 -> 5 starts [0, 48, 96, 144, 192]
extent=  70 tile=64 overlap=16 -> 2 starts [0, 6]
extent=  64 tile=64 overlap=16 -> 1 starts [0]
extent=  10 tile=64 overlap=16 -> 1 starts [0]
extent= 130 tile=64 overlap= 0 -> 3 starts [0, 64, 66]
tile math covers every pixel


In [17]:
# experiment: batched forward + Gaussian-weighted stitch on synthetic data
from segmentation_tools_pt.stitching import gaussian_weight_map

def _predict_tiled_proto(model, image, device, tile_size=64, overlap=16, inference_batch_size=2, task=None, activate=True):
    x = image.unsqueeze(0).to(device)
    H, W = x.shape[-2], x.shape[-1]
    ph, pw = max(0, tile_size - H), max(0, tile_size - W)
    if ph or pw: x = torch.nn.functional.pad(x, (0, pw, 0, ph))  # image smaller than a tile
    Hp, Wp = x.shape[-2], x.shape[-1]
    coords = [(y, xx) for y in _tile_starts_proto(Hp, tile_size, overlap) for xx in _tile_starts_proto(Wp, tile_size, overlap)]
    tiles = torch.stack([x[0, :, y:y + tile_size, xx:xx + tile_size] for y, xx in coords])  # (N, C, ts, ts)
    def act(logits):
        if not activate: return logits
        return torch.sigmoid(logits) if (logits.shape[1] == 1 or task == 'multilabel') else torch.softmax(logits, 1)
    wmap = gaussian_weight_map(tile_size, device=x.device)
    acc, wsum = None, x.new_zeros(1, Hp, Wp)
    for i in range(0, len(coords), inference_batch_size):  # <- the batching knob
        probs = act(model(tiles[i:i + inference_batch_size]))
        if acc is None: acc = probs.new_zeros(probs.shape[1], Hp, Wp)  # sized by OUTPUT channels
        for j, (y, xx) in enumerate(coords[i:i + inference_batch_size]):
            acc[:, y:y + tile_size, xx:xx + tile_size] += wmap * probs[j]
            wsum[:, y:y + tile_size, xx:xx + tile_size] += wmap
    return (acc / wsum)[:, :H, :W]  # (C_out, H, W)

import torch.nn as nn
mb = nn.Sequential(nn.Conv2d(1, 1, 3, padding=1), nn.Sigmoid())
img = torch.rand(1, 40, 50)
p1  = _predict_tiled_proto(mb, img, 'cpu', inference_batch_size=1)
p2  = _predict_tiled_proto(mb, img, 'cpu', inference_batch_size=2)
p99 = _predict_tiled_proto(mb, img, 'cpu', inference_batch_size=999)
assert p2.shape == (1, 40, 50)
assert torch.equal(p1, p2) and torch.equal(p2, p99), 'batch-size invariance failed'
assert 0 <= p2.min() and p2.max() <= 1
# verify: image smaller than a tile; multilabel head; logits passthrough
assert _predict_tiled_proto(mb, torch.rand(1, 10, 12), 'cpu', tile_size=64).shape == (1, 10, 12)
pm = _predict_tiled_proto(nn.Conv2d(1, 3, 3, padding=1), img, 'cpu', task='multilabel')
assert pm.shape == (3, 40, 50) and 0 <= pm.min() and pm.max() <= 1
lo = _predict_tiled_proto(nn.Conv2d(1, 1, 3, padding=1), img, 'cpu', activate=False)
assert bool(lo.min() < 0 or lo.max() > 1)
print('prototype verified: batch-invariant, padded, multilabel, logits passthrough')

prototype verified: batch-invariant, padded, multilabel, logits passthrough


In [18]:
#| export
def tile_starts(
    extent: int,     # image extent along one axis (H or W), already padded up if smaller than tile_size
    tile_size: int,  # tile side along the same axis
    overlap: int,    # overlap between adjacent tiles along the same axis
) -> List[int]:
    """Clamped sliding-window starts covering `[0, extent)`: 0 <= overlap < tile_size required.

    The last start is shifted back so every tile fits inside the extent — no output pixels
    are left uncovered and no tile runs past the edge.

    Example:
        >>> tile_starts(256, 64, 16)
        [0, 48, 96, 144, 192]
    """
    if not (0 <= overlap < tile_size):
        raise ValueError(f'overlap must satisfy 0 <= overlap < tile_size ({tile_size}), got {overlap}')
    stride = tile_size - overlap
    if extent <= tile_size: return [0]
    n = -(-(extent - tile_size) // stride) + 1  # ceil((extent - tile_size) / stride) + 1 windows
    return sorted({min(i * stride, extent - tile_size) for i in range(n)})

@torch.no_grad()
def predict_tiled(
    model,                          # trained model returning (B, C, H, W) logits
    image: torch.Tensor,            # a single (C, H, W) input — any H, W (smaller images are zero-padded up)
    device: torch.device,           # cpu or cuda
    tile_size: int = 256,           # tile side fed to the model
    overlap: int = 32,              # adjacent-tile overlap; Gaussian blending stitches tiles back
    inference_batch_size: int = 8,  # tiles per forward — raise for throughput, lower if OOM
    activate: bool = True,          # apply sigmoid/softmax to logits (same as `predict`)
    task: str = None,               # 'binary' | 'multiclass' | 'multilabel'; None infers like `predict`
) -> torch.Tensor:
    """Return `(C, H, W)` probabilities for a large image via batched sliding-window tiles.

    Tiles are forwarded in chunks of `inference_batch_size` (the `inference.inference_batch_size`
    config knob) and stitched with Gaussian-weighted blending. `inference_batch_size=1` gives
    bit-identical output to any larger value — the knob trades GPU memory against throughput
    only, never accuracy. Activation semantics match `predict`.

    Example:
        >>> prob = predict_tiled(model, torch.rand(1, 1152, 1632), dev, tile_size=256, overlap=32)
        >>> mask = (prob[0] >= 0.5).long()
    """
    if inference_batch_size < 1:
        raise ValueError(f'inference_batch_size must be >= 1, got {inference_batch_size}')
    model.eval()
    x = image.unsqueeze(0).to(device)
    H, W = x.shape[-2], x.shape[-1]
    ph, pw = max(0, tile_size - H), max(0, tile_size - W)
    if ph or pw:
        x = torch.nn.functional.pad(x, (0, pw, 0, ph), mode='constant', value=0)
    Hp, Wp = x.shape[-2], x.shape[-1]
    coords = [(y, xx) for y in tile_starts(Hp, tile_size, overlap)
              for xx in tile_starts(Wp, tile_size, overlap)]  # row-major, fixed order
    tiles = torch.stack([x[0, :, y:y + tile_size, xx:xx + tile_size] for y, xx in coords])  # (N, C, ts, ts)

    wmap = gaussian_weight_map(tile_size, device=x.device)
    acc, wsum = None, x.new_zeros(1, Hp, Wp)
    for i in range(0, len(coords), inference_batch_size):
        probs = _activate(model(tiles[i:i + inference_batch_size]), task, activate)
        if acc is None: acc = probs.new_zeros(probs.shape[1], Hp, Wp)  # output-channel aware
        for j, (y, xx) in enumerate(coords[i:i + inference_batch_size]):
            acc[:, y:y + tile_size, xx:xx + tile_size] += wmap * probs[j]
            wsum[:, y:y + tile_size, xx:xx + tile_size] += wmap
    return (acc / wsum)[:, :H, :W]  # (C_out, H, W)

In [19]:
#| test
# test: predict_tiled on synthetic data (same checks as the experiment cells, via exports)
import torch.nn as nn

mb = nn.Sequential(nn.Conv2d(1, 1, 3, padding=1), nn.Sigmoid())
img = torch.rand(1, 40, 50)
p1  = predict_tiled(mb, img, dev, inference_batch_size=1)
p2  = predict_tiled(mb, img, dev, inference_batch_size=2)
p99 = predict_tiled(mb, img, dev, inference_batch_size=999)
assert p2.shape == (1, 40, 50)
assert torch.equal(p1, p2) and torch.equal(p2, p99), 'batch-size invariance failed'
assert 0 <= p2.min() and p2.max() <= 1
assert tile_starts(256, 64, 16) == [0, 48, 96, 144, 192]
# image smaller than one tile; multichannel input; multiclass / multilabel heads; logits passthrough
assert predict_tiled(mb, torch.rand(1, 10, 12), dev, tile_size=64).shape == (1, 10, 12)
assert predict_tiled(nn.Conv2d(3, 1, 3, padding=1), torch.rand(3, 40, 50), dev).shape == (1, 40, 50)
assert predict_tiled(nn.Conv2d(1, 4, 3, padding=1), img, dev).shape == (4, 40, 50)
pm = predict_tiled(nn.Conv2d(1, 3, 3, padding=1), img, dev, task='multilabel')
assert pm.shape == (3, 40, 50) and 0 <= pm.min() and pm.max() <= 1
lo = predict_tiled(nn.Conv2d(1, 1, 3, padding=1), img, dev, activate=False)
assert bool(lo.min() < 0 or lo.max() > 1)
# clear failures on bad knobs
for bad in ({'overlap': 256}, {'inference_batch_size': 0}):
    try:
        predict_tiled(mb, img, dev, **bad); raise AssertionError(f'no error for {bad}')
    except ValueError:
        pass

In [20]:
#| hide
import nbdev; nbdev.nbdev_export()